# A · Load bookings from `data/raw/bookings`

Legacy CSV is read as Latin-1 at booking-test grain. Nova JSONL is read as UTF-8, nested tests are flattened, and timestamps are converted to Asia/Kolkata. The supplied sample is gzip-compressed; RAR extraction is supported when RAR files are present.

In [1]:
from __future__ import annotations
import gzip, io, json, pathlib, tempfile
import pandas as pd
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
BOOKINGS_ROOT = ROOT / 'data' / 'raw' / 'bookings'
CLEAN_ROOT = ROOT / 'data' / 'clean'
CLEAN_ROOT.mkdir(parents=True, exist_ok=True)
LEGACY_DTYPES = {'booking_id': 'string', 'city': 'category', 'clinic_id': 'category', 'channel': 'category', 'source': 'category', 'patient_id': 'string', 'patient_name': 'string', 'test_code': 'category', 'test_price': 'Int32', 'home_fee': 'Int32'}

def discover(root):
    rows = []
    for p in sorted(root.rglob('*')):
        if p.is_file() and p.suffix.lower() in {'.rar', '.gz', '.csv', '.jsonl'}:
            kind = 'archive' if p.suffix.lower() == '.rar' else ('nova' if '.jsonl' in p.name.lower() else 'legacy')
            rows.append({'path': p, 'kind': kind, 'bytes': p.stat().st_size})
    return pd.DataFrame(rows)
files = discover(BOOKINGS_ROOT)
files.assign(path=files.path.map(lambda p: p.relative_to(ROOT)))

,path,kind,bytes
0,data\raw\bookings\legacy\bengaluru\bookings_20...,legacy,215062
1,data\raw\bookings\legacy\chennai\bookings_2026...,legacy,158526
2,data\raw\bookings\legacy\jaipur\bookings_2026-...,legacy,80601
3,data\raw\bookings\nova\chennai\bookings_2026-0...,nova,145555
4,data\raw\bookings\nova\jaipur\bookings_2026-04...,nova,71895


In [2]:
rar_tmp = tempfile.TemporaryDirectory(prefix='kalpa-bookings-')
rar_files = files.loc[files.kind.eq('archive'), 'path'].tolist() if not files.empty else []
extracted = []
if rar_files:
    try:
        import importlib
        rarfile = importlib.import_module('rarfile')
    except ImportError as exc:
        raise RuntimeError('RAR input found: install rarfile and a RAR backend before loading.') from exc
    for archive_path in rar_files:
        target = pathlib.Path(rar_tmp.name) / archive_path.stem
        target.mkdir(parents=True, exist_ok=True)
        with rarfile.RarFile(archive_path) as archive:
            archive.extractall(target)
        extracted.extend(p for p in target.rglob('*') if p.is_file())
input_paths = files.loc[files.kind.ne('archive'), 'path'].tolist() + extracted

def text_handle(path, encoding):
    return gzip.open(path, 'rt', encoding=encoding) if path.name.lower().endswith('.gz') else path.open('rt', encoding=encoding)

def read_legacy(path):
    with text_handle(path, 'latin-1') as handle:
        lines = handle.read().splitlines()
    header = lines[0]
    body = [line for line in lines[1:] if line != header]
    frame = pd.read_csv(io.StringIO('\n'.join([header, *body])), dtype=LEGACY_DTYPES)
    for column in ('created_at', 'slot_at'):
        frame[column] = pd.to_datetime(frame[column], format='%d/%m/%Y %H:%M', errors='coerce')
    frame['booking_id'] = frame['booking_id'].str.strip()
    frame['source_file'], frame['system'] = path.name, 'legacy'
    frame.attrs['repeated_header_lines'] = len(lines) - 1 - len(body)
    return frame

def read_nova(path):
    rows = []
    with text_handle(path, 'utf-8') as handle:
        for line in handle:
            record = json.loads(line)
            for test in record['tests']:
                rows.append({'booking_id': record['booking_id'], 'created_at': record['created_at'], 'slot_at': record['slot_at'], 'city': record['city'], 'clinic_id': record['clinic_id'], 'channel': record['channel'].lower(), 'source': record['source'].lower(), 'patient_id': record['patient']['patient_id'], 'patient_name': pd.NA, 'test_code': test['code'], 'test_price': test['price'], 'home_fee': record['home_fee'], 'rescheduled_from': record.get('rescheduled_from'), 'status_at_export': record.get('status_at_export')})
    frame = pd.DataFrame(rows)
    for column in ('created_at', 'slot_at'):
        frame[column] = pd.to_datetime(frame[column], utc=True).dt.tz_convert('Asia/Kolkata').dt.tz_localize(None)
    frame['source_file'], frame['system'] = path.name, 'nova'
    return frame

In [3]:
frames = []
for path in sorted(input_paths, key=str):
    if '.jsonl' in path.name.lower():
        frames.append(read_nova(path))
    elif '.csv' in path.name.lower():
        frames.append(read_legacy(path))
    else:
        raise ValueError(f'Unsupported extracted booking file: {path}')
columns = []
for frame in frames:
    columns.extend(column for column in frame.columns if column not in columns)
for frame in frames:
    for column in set(columns).difference(frame.columns):
        frame[column] = pd.NA
bookings_all = pd.concat([frame[columns] for frame in frames], ignore_index=True)
bookings_all['city'] = bookings_all['city'].astype('string').replace({'Bangalore': 'Bengaluru'})
repeated_headers = sum(frame.attrs.get('repeated_header_lines', 0) for frame in frames)
assert not bookings_all.empty and bookings_all['booking_id'].notna().all()
assert bookings_all['created_at'].notna().all() and bookings_all['slot_at'].notna().all()
output_path = CLEAN_ROOT / 'bookings_all.csv.gz'
bookings_all.to_csv(output_path, index=False, compression='gzip')
print(f'PASS: {len(bookings_all):,} booking-test rows; {bookings_all.booking_id.nunique():,} bookings; {repeated_headers} repeated headers dropped')
print(f'wrote {output_path}')
bookings_all.head()

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_17900\3988241335.py:15: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  bookings_all = pd.concat([frame[columns] for frame in frames], ignore_index=True)


PASS: 58,362 booking-test rows; 25,069 bookings; 0 repeated headers dropped
wrote c:\Users\Lenovo\Desktop\Pure_Kalpa_Data_Folder\data\clean\bookings_all.csv.gz


,booking_id,created_at,slot_at,city,clinic_id,channel,source,patient_id,patient_name,test_code,test_price,home_fee,source_file,system,rescheduled_from,status_at_export
0,BK-BLR-202503-014680,2025-03-21 03:40:00,2025-03-23 07:00:00,Bengaluru,BLR-03,clinic,app,PBLR0009350,Diya Shah,T005,700,0,bookings_2025-03.csv.gz,legacy,NaN,NaN
1,BK-BLR-202503-014681,2025-03-03 09:58:00,2025-03-04 16:51:00,Bengaluru,BLR-01,clinic,walkin,PBLR0006384,Ritika Gupta,T008,1100,0,bookings_2025-03.csv.gz,legacy,NaN,NaN
2,BK-BLR-202503-014682,2025-03-21 04:34:00,2025-03-21 07:44:00,Bengaluru,BLR-01,home,phone,PBLR0029923,Ishaan Nair,T001,350,150,bookings_2025-03.csv.gz,legacy,NaN,NaN
3,BK-BLR-202503-014682,2025-03-21 04:34:00,2025-03-21 07:44:00,Bengaluru,BLR-01,home,phone,PBLR0029923,Ishaan Nair,T008,1100,150,bookings_2025-03.csv.gz,legacy,NaN,NaN
4,BK-BLR-202503-014682,2025-03-21 04:34:00,2025-03-21 07:44:00,Bengaluru,BLR-01,home,phone,PBLR0029923,Ishaan Nair,T005,700,150,bookings_2025-03.csv.gz,legacy,NaN,NaN


In [4]:
bookings_all.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 58362 entries, 0 to 58361
Data columns (total 16 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   booking_id        58362 non-null  object        
 1   created_at        58362 non-null  datetime64[ns]
 2   slot_at           58362 non-null  datetime64[ns]
 3   city              58362 non-null  string        
 4   clinic_id         58362 non-null  object        
 5   channel           58362 non-null  object        
 6   source            58362 non-null  object        
 7   patient_id        58362 non-null  object        
 8   patient_name      37658 non-null  string        
 9   test_code         58334 non-null  object        
 10  test_price        58362 non-null  Int64         
 11  home_fee          58362 non-null  Int64         
 12  source_file       58362 non-null  object        
 13  system            58362 non-null  object        
 14  rescheduled_from  1169